# 01 · Data Inspection
**Project:** Digital Advertising Performance Intelligence

**Goal of this notebook:** load the raw CSV exactly as provided, understand its structure, and list the data-quality problems that notebook 02 will fix.

**Run order** : **01 → 02 → 03 → 04** (run each notebook top to bottom from the `notebooks/` folder).

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.4f}".format)

RAW_PATH = Path("..") / "data" / "raw" / "online_advertising_performance_data.csv"
df = pd.read_csv(RAW_PATH)
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
df.head()

Rows: 15,408 | Columns: 14


,month,day,campaign_number,user_engagement,banner,placement,displays,cost,clicks,revenue,post_click_conversions,post_click_sales_amount,Unnamed: 12,Unnamed: 13
0,April,1,camp 1,High,160 x 600,abc,4,0.0060,0,0.0000,0,0.0000,NaN,NaN
1,April,1,camp 1,High,160 x 600,def,20170,26.7824,158,28.9717,23,"1,972.4602",NaN,NaN
2,April,1,camp 1,High,160 x 600,ghi,14701,27.6304,158,28.9771,78,"2,497.2636",NaN,NaN
3,April,1,camp 1,High,160 x 600,mno,171259,216.8750,1796,329.4518,617,"24,625.3234",NaN,NaN
4,April,1,camp 1,Low,160 x 600,def,552,0.0670,1,0.1834,0,0.0000,NaN,NaN


## 1. Columns and data types

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15408 entries, 0 to 15407
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   month                    15408 non-null  str    
 1   day                      15408 non-null  int64  
 2   campaign_number          15408 non-null  str    
 3   user_engagement          15408 non-null  str    
 4   banner                   15408 non-null  str    
 5   placement                14995 non-null  str    
 6   displays                 15408 non-null  int64  
 7   cost                     15408 non-null  float64
 8   clicks                   15408 non-null  int64  
 9   revenue                  15408 non-null  float64
 10  post_click_conversions   15408 non-null  int64  
 11  post_click_sales_amount  15408 non-null  float64
 12  Unnamed: 12              0 non-null      float64
 13  Unnamed: 13              0 non-null      float64
dtypes: float64(5), int64(4), str(5)
m

**What this shows:** the file has 14 columns, but the last two are called `Unnamed: 12` and `Unnamed: 13` and contain no data. They come from trailing commas in the CSV and carry no information. The 12 real columns have sensible types (numbers are numeric, labels are text).

In [3]:
column_guide = pd.DataFrame({
    "column": ["month", "day", "campaign_number", "user_engagement", "banner", "placement",
               "displays", "cost", "clicks", "revenue", "post_click_conversions", "post_click_sales_amount"],
    "type": ["category", "category", "category", "category (Low/Medium/High)", "category (banner size)", "category (site code)",
             "count", "money", "count", "money", "count", "money"],
})
column_guide

,column,type
0,month,category
1,day,category
2,campaign_number,category
3,user_engagement,category (Low/Medium/High)
4,banner,category (banner size)
5,placement,category (site code)
6,displays,count
7,cost,money
8,clicks,count
9,revenue,money


## 2. Basic statistics

In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
day,"15,408.0000",15.5189,8.7409,1.0000,8.0000,15.0000,23.0000,31.0000
displays,"15,408.0000","15,512.5730","44,392.3929",0.0000,78.0000,"1,182.0000","8,960.2500","455,986.0000"
cost,"15,408.0000",11.3703,45.3695,0.0000,0.0240,0.3398,2.5362,556.7048
clicks,"15,408.0000",161.7885,728.2769,0.0000,0.0000,6.0000,53.0000,"14,566.0000"
revenue,"15,408.0000",17.9299,96.7818,0.0000,0.0000,0.4839,3.8398,"2,096.2116"
post_click_conversions,"15,408.0000",42.3006,213.6857,0.0000,0.0000,0.0000,3.0000,"3,369.0000"
post_click_sales_amount,"15,408.0000","2,123.2881","10,523.0296",0.0000,0.0000,0.0000,163.3512,"199,930.3180"
Unnamed: 12,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Unnamed: 13,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df.select_dtypes(exclude="number").describe().T

,count,unique,top,freq
month,15408,3,April,6269
campaign_number,15408,3,camp 3,6917
user_engagement,15408,3,Medium,5489
banner,15408,8,728 x 90,3142
placement,14995,5,mno,4501


**What this shows:** values are heavily right-skewed (the mean is far above the median for `displays`, `cost`, `clicks`, `revenue`). Many rows have zero clicks or zero conversions, which is normal for small ad placements. There are no negative values.

## 3. Missing values

In [6]:
missing = pd.DataFrame({"missing": df.isna().sum(), "percent": df.isna().mean() * 100})
missing[missing["missing"] > 0]

,missing,percent
placement,413,2.6804
Unnamed: 12,15408,100.0000
Unnamed: 13,15408,100.0000


In [7]:
print("Missing placement by campaign and month:")
df[df["placement"].isna()].groupby(["campaign_number", "month"]).size().rename("rows")

Missing placement by campaign and month:


campaign_number  month
camp 1           April    61
                 June     58
                 May      42
camp 2           April    55
camp 3           April    77
                 June     68
                 May      52
Name: rows, dtype: int64

**What this shows:** only `placement` has real missing values (413 rows, 2.7%). The two unnamed columns are 100% empty.

## 4. Duplicate rows

In [8]:
dup_mask = df.duplicated(keep=False)
print("Rows that are part of a duplicate pair:", dup_mask.sum())
print("Extra copies (would be removed):", df.duplicated().sum())
df[dup_mask].sort_values(["month", "day", "campaign_number", "banner", "placement"])

Rows that are part of a duplicate pair: 10
Extra copies (would be removed): 5


,month,day,campaign_number,user_engagement,banner,placement,displays,cost,clicks,revenue,post_click_conversions,post_click_sales_amount,Unnamed: 12,Unnamed: 13
5,April,1,camp 1,Low,160 x 600,ghi,16,0.0249,0,0.0000,0,0.0000,NaN,NaN
15403,April,1,camp 1,Low,160 x 600,ghi,16,0.0249,0,0.0000,0,0.0000,NaN,NaN
6,April,1,camp 1,Low,160 x 600,mno,2234,0.4044,10,1.8347,3,101.7494,NaN,NaN
15404,April,1,camp 1,Low,160 x 600,mno,2234,0.4044,10,1.8347,3,101.7494,NaN,NaN
15196,June,29,camp 1,High,800 x 250,ghi,1,0.0157,0,0.0000,0,0.0000,NaN,NaN
15405,June,29,camp 1,High,800 x 250,ghi,1,0.0157,0,0.0000,0,0.0000,NaN,NaN
15197,June,29,camp 1,High,800 x 250,mno,4,0.0123,0,0.0000,0,0.0000,NaN,NaN
15406,June,29,camp 1,High,800 x 250,mno,4,0.0123,0,0.0000,0,0.0000,NaN,NaN
15198,June,29,camp 3,High,240 x 400,def,1209,0.3184,2,0.1115,3,110.4224,NaN,NaN
15407,June,29,camp 3,High,240 x 400,def,1209,0.3184,2,0.1115,3,110.4224,NaN,NaN


In [9]:
# Is one row supposed to be unique per month/day/campaign/engagement/banner/placement?
grain = ["month", "day", "campaign_number", "user_engagement", "banner", "placement"]
deduped = df.drop_duplicates()
print("Duplicated grain keys in raw data:      ", df.duplicated(grain, keep=False).sum())
print("Duplicated grain keys after de-duplicating:", deduped.duplicated(grain, keep=False).sum())
print("Positions of the extra copies in the file:", list(df.index[df.duplicated()]))

Duplicated grain keys in raw data:       10
Duplicated grain keys after de-duplicating: 0


Positions of the extra copies in the file: [15403, 15404, 15405, 15406, 15407]


**What this shows:** 5 rows are exact copies of earlier rows, and all 5 sit at the very end of the file. After removing them, every row is unique at the level *month / day / campaign / engagement / banner / placement*, so the copies are almost certainly accidental repeats, not real data.

## 5. Unique values in category columns

In [10]:
for col in ["month", "campaign_number", "user_engagement", "banner", "placement"]:
    print(f"{col}: {sorted(df[col].dropna().unique())}")
print()
print(df.groupby("month")["day"].agg(["min", "max", "nunique"]))
print()
print("Rows per campaign and month:")
df.groupby(["campaign_number", "month"]).size().unstack()

month: ['April', 'June', 'May']
campaign_number: ['camp 1', 'camp 2', 'camp 3']
user_engagement: ['High', 'Low', 'Medium']
banner: ['160 x 600', '240 x 400', '300 x 250', '468 x 60', '580 x 400', '670 x 90', '728 x 90', '800 x 250']
placement: ['abc', 'def', 'ghi', 'jkl', 'mno']



       min  max  nunique
month                   
April    1   30       30
June     1   30       30
May      1   31       31

Rows per campaign and month:


month,April,June,May
campaign_number,,,
camp 1,"2,315.0000","2,216.0000","2,346.0000"
camp 2,"1,614.0000",NaN,NaN
camp 3,"2,340.0000","2,218.0000","2,359.0000"


**What this shows:** labels are consistent (no spelling variants or stray spaces). Dates are valid (April and June have 30 days, May has 31). **Important:** there is no year, and `camp 2` only has data for **April**, so month-over-month trends can only compare camp 1 and camp 3.

## 6. Logical consistency checks

In [11]:
checks = {
    "clicks > displays": df["clicks"] > df["displays"],
    "post_click_conversions > clicks": df["post_click_conversions"] > df["clicks"],
    "displays > 0 but cost = 0": (df["displays"] > 0) & (df["cost"] == 0),
    "cost > 0 but displays = 0": (df["cost"] > 0) & (df["displays"] == 0),
    "sales > 0 but conversions = 0": (df["post_click_sales_amount"] > 0) & (df["post_click_conversions"] == 0),
    "conversions > 0 but sales = 0": (df["post_click_conversions"] > 0) & (df["post_click_sales_amount"] == 0),
    "revenue > 0 but clicks = 0": (df["revenue"] > 0) & (df["clicks"] == 0),
}
pd.Series({name: int(mask.sum()) for name, mask in checks.items()}, name="rows").to_frame()

,rows
clicks > displays,2
post_click_conversions > clicks,329
displays > 0 but cost = 0,57
cost > 0 but displays = 0,15
sales > 0 but conversions = 0,19
conversions > 0 but sales = 0,0
revenue > 0 but clicks = 0,0


In [12]:
# A closer look at the largest group: conversions greater than clicks
over = df[df["post_click_conversions"] > df["clicks"]]
print("Share by campaign:")
print(over["campaign_number"].value_counts())
over.head()

Share by campaign:
campaign_number
camp 1    312
camp 3     15
camp 2      2
Name: count, dtype: int64


,month,day,campaign_number,user_engagement,banner,placement,displays,cost,clicks,revenue,post_click_conversions,post_click_sales_amount,Unnamed: 12,Unnamed: 13
244,April,2,camp 1,High,468 x 60,mno,3432,1.7688,9,1.6259,17,"1,808.8476",NaN,NaN
440,April,3,camp 1,Low,300 x 250,mno,2898,0.5847,9,1.6269,14,339.4011,NaN,NaN
448,April,3,camp 1,High,468 x 60,mno,4006,1.9898,4,0.7221,9,143.2366,NaN,NaN
865,April,5,camp 1,High,468 x 60,ghi,495,0.1924,1,0.1814,5,75.3821,NaN,NaN
867,April,5,camp 1,High,468 x 60,mno,3559,2.1829,10,1.8145,16,"1,624.7739",NaN,NaN


## Summary of data-quality problems

| # | Problem | Size | Planned handling (notebook 02) |
|---|---|---|---|
| 1 | Two completely empty unnamed columns | 2 columns | Drop |
| 2 | Missing `placement` | 413 rows (2.7%) | Fill with `Unknown` (keep the spend and clicks) |
| 3 | Exact duplicate rows | 5 rows | Remove the extra copies |
| 4 | `clicks > displays` | 2 rows | Keep, add a flag |
| 5 | `post_click_conversions > clicks` | 329 rows | Keep, add a flag; treat conversion-rate results with care |
| 6 | Cost with no displays / displays with no cost | 15 / 57 rows | Keep, add a flag (tiny amounts) |
| 7 | Sales with no conversions | 19 rows | Keep, flag only |
| 8 | No year column; `camp 2` only in April | - | Document as a limitation |

Nothing is deleted except the 5 exact duplicates and the 2 empty columns.